In [163]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, InputLayer
import os
import random
import h5py
from scipy.signal import decimate

In [164]:
import random
import numpy as np
import tensorflow as tf

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)


In [165]:

def get_dataset_name(file_name_with_dir):
    
    filename_without_dir = file_name_with_dir.split('/')[-1]
    print(filename_without_dir)
    temp = filename_without_dir.split('_')[:-1]
    print(temp)
    dataset_name = "_".join(temp)
    return dataset_name
filename_path="./data/Intra/train/rest_105923_1.h5"
with h5py. File (filename_path , 'r') as f :
    dataset_name = get_dataset_name(filename_path)
    print(dataset_name)
    matrix = f.get(dataset_name)[()]
    print(type(matrix ))
    print(matrix.shape)

rest_105923_1.h5
['rest', '105923']
rest_105923
<class 'numpy.ndarray'>
(248, 35624)


In [166]:
def z_score_normalization(data):
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    return (data - mean) / std

In [167]:
def segment_data(data, label, window_size=500, stride=500):
    segments = []
    labels = []
    for start in range(0, data.shape[1] - window_size + 1, stride):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)
        labels.append(label)
    return segments, labels

In [168]:
def infer_label_from_filename(filename, label_map):
    filename = filename.lower().replace('\\', '/')
    basename = os.path.basename(filename)
    
    for key in label_map:
        if key in basename:
            return label_map[key]
    
    raise ValueError(f"Could not infer label from filename: {filename}")

In [169]:
def load_and_preprocess(filepath, label_map, num_chunks=50, downsample_factor=20):

    task_label = infer_label_from_filename(filepath, label_map)

    with h5py.File(filepath, 'r') as f:
        datasetname = list(f.keys())[0]
        data = f[datasetname][()]  # Shape: (248, T)

    # Downsample
    data = decimate(data, q=downsample_factor, axis=1)  # e.g., 35624 -> ~1781

    total_length = data.shape[1]
    chunk_length = total_length // num_chunks

    segments = []
    for i in range(num_chunks):
        start = i * chunk_length
        end = start + chunk_length
        if end > total_length:
            break

        window = data[:, start:end]

        # z-score normalization per channel
        mean = window.mean(axis=1, keepdims=True)
        std = window.std(axis=1, keepdims=True)
        window = (window - mean) / (std + 1e-8)

        segments.append(window[..., np.newaxis])  # (248, chunk_len, 1)

    labels = [task_label] * len(segments)
    return segments, labels


In [170]:
def data_generator(filepaths, label_map, batch_size, file_batch_size=8):
    while True:
        random.shuffle(filepaths)

        for i in range(0, len(filepaths), file_batch_size):
            file_batch = filepaths[i:i+file_batch_size]

            all_segments, all_labels = [], []
            for filepath in file_batch:
                segments, labels = load_and_preprocess(filepath, label_map)
                all_segments.extend(segments)
                all_labels.extend(labels)

            X = np.array(all_segments)
            y = np.array(all_labels)

            # Shuffle batch data
            idx = np.random.permutation(len(y))
            X, y = X[idx], y[idx]

            # Yield in batches
            for j in range(0, len(X), batch_size):
                yield X[j:j+batch_size], y[j:j+batch_size]


In [171]:
# Parameters
train_dir = './data/Cross/train'

# Load all file paths and shuffle
all_filepaths = [
    os.path.normpath(os.path.join(train_dir, fname))
    for fname in os.listdir(train_dir)
    if fname.endswith('.h5')
]
np.random.shuffle(all_filepaths)

# Label map
label_map = {
    'rest': 0,
    'math': 1,
    'story': 1,
    'story_math': 1,         
    'working_memory': 2,
    'memory': 2,
    'motor': 3
}


In [172]:
segments, _ = load_and_preprocess(all_filepaths[0], label_map)
print(f"Segment shape: {segments[0].shape}")

Segment shape: (248, 35, 1)


In [173]:
from keras import layers, models, regularizers
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
def build_cnn(input_shape=segments[0].shape, num_classes=4, l2_lambda=0.01): # shape = (248, 35, 1)
    model = Sequential([
        InputLayer(input_shape=input_shape),

        Conv2D(32, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),
 

        Conv2D(64, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),


        Conv2D(128, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),


        Flatten(),
        Dropout(0.6),
        Dense(128, activation='relu', kernel_regularizer=l2(l2_lambda)),
        # Dropout(0.7),
        Dense(num_classes, activation='softmax')
    ])
    
   
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [182]:
def load_test_data(test_filepaths, label_map, downsample_factor=20):
    all_segments = []
    all_labels = []

    for filepath in test_filepaths:
        segments, labels = load_and_preprocess(
            filepath, label_map,
            downsample_factor=downsample_factor
        )
        all_segments.extend(segments)
        all_labels.extend(labels)

    X_test = np.array(all_segments)
    y_test = np.array(all_labels)
    return X_test, y_test


In [ ]:
import os
import numpy as np
from sklearn.utils import shuffle

def chunked_training(model, train_filepaths, label_map, chunk_size=50, batch_size=16):
    num_chunks = len(train_filepaths) // chunk_size

    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch+1}/{epochs} ===")
        shuffled_files = shuffle(train_filepaths)

        for i in range(num_chunks):
            chunk_files = shuffled_files[i * chunk_size : (i + 1) * chunk_size]
            print(f"\nTraining on files {i * chunk_size + 1} to {(i + 1) * chunk_size}")

            # Load and preprocess only this chunk
            X_batch, y_batch = load_test_data(chunk_files, label_map)

            # Fit model on this chunk
            model.fit(X_batch, y_batch, batch_size=batch_size, epochs=1, verbose=1)

    return model


In [176]:
import glob

train_folder = './data/Cross/train'
train_filepaths = glob.glob(os.path.join(train_folder, '*.h5'))
train_filepaths = [os.path.normpath(p) for p in train_filepaths]

In [177]:
def count_total_segments(filepaths, label_map):
    total = 0
    for fp in filepaths:
        segments, _ = load_and_preprocess(fp, label_map)
        total += len(segments)
    return total

In [178]:
from sklearn.model_selection import train_test_split

train_files, val_files = train_test_split(filepaths, test_size=0.2, random_state=42)


In [179]:
from tensorflow.keras.callbacks import EarlyStopping

# Build model
model = build_cnn()

# determine steps per epoch
train_total_segments = count_total_segments(train_files, label_map)
train_steps_per_epoch = train_total_segments // batch_size
train_gen = data_generator(train_files, label_map, batch_size=batch_size)

val_total_segments = count_total_segments(val_files, label_map)
val_steps_per_epoch = val_total_segments // batch_size
val_gen = data_generator(val_files, label_map, batch_size=batch_size)

early_stop = EarlyStopping(
    monitor='val_loss',       
    patience=3,               
    restore_best_weights=True
)

# Fit model
model.fit(
    train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=val_gen,
    validation_steps=val_steps_per_epoch
    # callbacks=[early_stop]
)

Epoch 1/15
159/159 [==============================] - 69s 432ms/step - loss: 3.7092 - accuracy: 0.5798 - val_loss: 3.3256 - val_accuracy: 0.3063
Epoch 2/15
159/159 [==============================] - 72s 458ms/step - loss: 2.4217 - accuracy: 0.6551 - val_loss: 2.7993 - val_accuracy: 0.1562
Epoch 3/15
159/159 [==============================] - 76s 476ms/step - loss: 1.9211 - accuracy: 0.7356 - val_loss: 4.8762 - val_accuracy: 0.2313
Epoch 4/15
159/159 [==============================] - 71s 450ms/step - loss: 1.6654 - accuracy: 0.7664 - val_loss: 4.4949 - val_accuracy: 0.1531
Epoch 5/15
159/159 [==============================] - 82s 517ms/step - loss: 1.4292 - accuracy: 0.8047 - val_loss: 17.2830 - val_accuracy: 0.1547
Epoch 6/15
159/159 [==============================] - 76s 481ms/step - loss: 1.2902 - accuracy: 0.8268 - val_loss: 40.6049 - val_accuracy: 0.1562
Epoch 7/15
159/159 [==============================] - 74s 464ms/step - loss: 1.2165 - accuracy: 0.8311 - val_loss: 12.5530 - val

In [183]:
import glob

for i in range(1, 4):
    # Collect test files
    test_folder = f"./data/Cross/test{i}"
    test_filepaths = glob.glob(os.path.join(test_folder, '*.h5'))
    test_filepaths = [os.path.normpath(p) for p in test_filepaths]

    # Load and preprocess test data
    X_test, y_test = load_test_data(test_filepaths, label_map)
    # Direct evaluation
    loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
    print(f"Test Accuracy: {accuracy}")

25/25 [==============================] - 3s 114ms/step - loss: 397.3072 - accuracy: 0.2500
Test Accuracy: 0.25
25/25 [==============================] - 3s 104ms/step - loss: 395.7004 - accuracy: 0.2500
Test Accuracy: 0.25
25/25 [==============================] - 3s 102ms/step - loss: 395.3394 - accuracy: 0.2500
Test Accuracy: 0.25
